# 01. Môi trường FlappyBird

Notebook này là module duy nhất của dự án được import `gymnasium`. Các phần còn lại (Q-Network, Experience Replay, agent) chỉ làm việc với mảng số, nên kiểm thử được bằng dữ liệu giả.

Nội dung:

1. Hằng số của bài toán.
2. Lớp `FlappyEnv` bọc `FlappyBird-v0`.
3. Hàm `play_episodes` và `summarize_scores` dùng để chấm điểm một policy.
4. Kiểm tra nhanh.

Chi tiết về thư viện nằm ở mục 1 và 2 của `Implementation_Notes.md`.

## 1. Hằng số

- Observation gồm 12 số đã chia tỉ lệ về khoảng −1 đến 1 (mục 2 của `Implementation_Notes.md`).
- Hai hành động: `IDLE = 0` (không làm gì) và `FLAP = 1` (vỗ cánh).

In [ ]:
import numpy as np
import gymnasium
import flappy_bird_gymnasium  # đăng ký "FlappyBird-v0" với gymnasium

ENV_ID = "FlappyBird-v0"
OBS_DIM = 12
N_ACTIONS = 2
IDLE, FLAP = 0, 1

## 2. Lớp `FlappyEnv`

Lớp này xử lý ba điểm dễ sai của thư viện:

- Mặc định thư viện dùng chế độ LIDAR (180 tia), nên phải tạo môi trường với `use_lidar=False`.
- Observation trả về kiểu `float64`; lớp đổi sang `float32` trước khi đưa cho mạng.
- Sau khi chim va chạm (`terminated`), thư viện vẫn cho gọi `step()` và chạy tiếp với con chim đã chết. Lớp này báo lỗi nếu gọi `step()` khi ván đã kết thúc, để các transition sai không lọt vào Experience Replay.

`step()` trả về `(obs, reward, terminated, truncated, score)`. `truncated` chỉ là `True` khi đặt `score_limit` và điểm đạt giới hạn; lúc đó chim chưa chết. `score` là số ống đã vượt, chỉ dùng để chấm điểm, không phải một phần của trạng thái.

Khi huấn luyện, để `render_mode=None`. Chế độ `"human"` bị khóa ở 30 khung hình/giây, chỉ dùng để xem agent chơi.

In [ ]:
class FlappyEnv:
    """Bọc FlappyBird-v0: observation float32, chặn step() sau khi ván đã kết thúc."""

    def __init__(self, score_limit=None, render_mode=None):
        self._env = gymnasium.make(ENV_ID, use_lidar=False,
                                   score_limit=score_limit, render_mode=render_mode)
        self._episode_over = True

    def reset(self, seed=None):
        obs, _ = self._env.reset(seed=seed)
        self._episode_over = False
        return obs.astype(np.float32)

    def step(self, action):
        if self._episode_over:
            raise RuntimeError("Ván đã kết thúc: phải gọi reset() trước khi gọi step().")
        obs, reward, terminated, truncated, info = self._env.step(int(action))
        self._episode_over = terminated or truncated
        return obs.astype(np.float32), float(reward), bool(terminated), bool(truncated), info["score"]

    def close(self):
        self._env.close()

## 3. Chấm điểm một policy

`play_episodes` cho một policy bất kỳ chơi nhiều ván và trả về số ống vượt được ở mỗi ván. Policy là một hàm nhận observation và trả về hành động, nên hàm này không phụ thuộc vào agent.

- Seed (số khởi tạo bộ sinh ngẫu nhiên, quyết định chuỗi khe) chỉ đặt ở ván đầu, nên cùng seed cho cùng chuỗi ván và các lần đánh giá so sánh được với nhau.
- `score_limit` là bắt buộc: một agent giỏi có thể chơi mãi, và giới hạn này bảo đảm hàm luôn kết thúc.

`summarize_scores` tính trung bình, trung vị, thấp nhất và cao nhất, vì điểm dao động rất mạnh giữa các ván.

In [ ]:
def play_episodes(policy, n_episodes, seed, score_limit):
    """Chơi n_episodes ván với policy(obs) -> action, trả về mảng số ống vượt được mỗi ván."""
    env = FlappyEnv(score_limit=score_limit)
    scores = []
    try:
        for episode in range(n_episodes):
            obs = env.reset(seed=seed if episode == 0 else None)
            episode_over = False
            while not episode_over:
                obs, _, terminated, truncated, score = env.step(policy(obs))
                episode_over = terminated or truncated
            scores.append(score)
    finally:
        env.close()
    return np.array(scores)


def summarize_scores(scores):
    return {
        "mean": float(np.mean(scores)),
        "median": float(np.median(scores)),
        "min": int(np.min(scores)),
        "max": int(np.max(scores)),
    }

## 4. Kiểm tra nhanh

Chạy vài ván ngắn để kiểm tra:

- Observation có đúng 12 số kiểu `float32`.
- Gọi `step()` sau khi chim chết thì báo lỗi.
- Cùng seed và cùng policy cho cùng kết quả.

In [ ]:
def _self_check_environment():
    env = FlappyEnv()
    rng = np.random.default_rng(0)
    obs = env.reset(seed=0)
    assert obs.shape == (OBS_DIM,) and obs.dtype == np.float32

    episode_over = False
    while not episode_over:
        obs, _, terminated, truncated, _ = env.step(FLAP if rng.random() < 0.5 else IDLE)
        episode_over = terminated or truncated
    try:
        env.step(IDLE)
        raise AssertionError("step() sau khi ván kết thúc phải báo lỗi")
    except RuntimeError:
        pass
    env.close()

    # Policy cố định: vỗ cánh khi chim xuống thấp hơn giữa màn hình (chỉ số 9 là độ cao chim)
    hold_middle = lambda o: FLAP if o[9] > 0.45 else IDLE
    first = play_episodes(hold_middle, 3, seed=0, score_limit=5)
    second = play_episodes(hold_middle, 3, seed=0, score_limit=5)
    assert len(first) == 3 and np.array_equal(first, second)
    print("01_environment: kiểm tra đạt.", summarize_scores(first))


_self_check_environment()